# M3 — train the student on a free Colab GPU

Fine-tunes one FinBERT encoder with three heads (category, materiality, direction) on the
teacher's labels, and reports fidelity against a held-out **time** period the model never saw.

This takes roughly **5 minutes on a T4** and about **3.5 hours on a laptop CPU**, which is the
whole reason for running it here.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. The next cell checks it.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session.\n"
        "Training on Colab's CPU is slower than your own laptop and not worth doing."
    )
print(torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 1. Get the code

Clones the public repo. Re-run this cell after pushing changes to pick them up.

In [ ]:
import os, shutil

REPO = 'https://github.com/Ron074/distillery-news.git'
if os.path.exists('distillery-news'):
    shutil.rmtree('distillery-news')
!git clone --depth 1 $REPO
%cd distillery-news
!pip install -q 'transformers>=4.46' 'scikit-learn>=1.4' 'pandas>=2.2,<3'
print('\nready')

## 2. Upload the labels and the sample

These are **not** in git: the labels are regenerable from the teacher and the corpus is large, so
committing them would bloat the repo for no benefit. Upload both from your machine:

- `data/labels/teacher_full.jsonl` — about 5.9 MB
- `data/samples/fnspid_clean_48040_seed42.csv` — about 10.2 MB

Pick both at once in the file dialog. If the upload keeps timing out, put the two files in Google
Drive instead and use the commented-out alternative below.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('data/labels', exist_ok=True)
os.makedirs('data/samples', exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    dest = 'data/labels' if name.endswith('.jsonl') else 'data/samples'
    shutil.move(name, f'{dest}/{name}')
    print(f'  {name} -> {dest}/')

# Alternative, if uploading is painful — put the two files in a Drive folder first:
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/distillery/teacher_full.jsonl data/labels/
# !cp /content/drive/MyDrive/distillery/fnspid_clean_48040_seed42.csv data/samples/

In [ ]:
# Sanity-check what arrived before spending GPU time on it.
import glob, json

labels = glob.glob('data/labels/*.jsonl')
samples = glob.glob('data/samples/*.csv')
assert labels, 'no .jsonl in data/labels — re-run the upload cell'
assert samples, 'no .csv in data/samples — re-run the upload cell'

n = sum(1 for _ in open(labels[0], encoding='utf-8'))
first = json.loads(open(labels[0], encoding='utf-8').readline())
print(f'labels : {labels[0]}  ({n:,} rows)')
print(f'sample : {samples[0]}')
print(f'fields : {sorted(first)}')
SAMPLE = samples[0]

## 3. Train the main model

Train on years before 2018, tune on 2018, and test on **2019 onward** — a period the model never
sees. Near-duplicate headlines are dropped before splitting so templated items cannot sit on both
sides and inflate the score through recognition rather than learning.

Class weights are applied per head. Without them, `high` materiality at 3.3% of the corpus means a
model that never predicts it still scores well — so `high` recall is printed separately rather than
being averaged away by macro-F1.

In [ ]:
!cd code && python train_student.py \
    --labels full \
    --sample ../$SAMPLE \
    --encoder yiyanghkust/finbert-pretrain \
    --epochs 3 --batch-size 32 --log-every 100 \
    --save-dir ../checkpoints/student_3head \
    --tag 3head

## 4. Read the result

**Fidelity is agreement with the teacher, not correctness.** The teacher agrees with *itself* only
93.0% of the time across 500 repeated headlines, so that is the ceiling — a student at 85% is
closing most of the reachable gap, not falling 15 points short of perfect. Quote the two together.

In [ ]:
import json

run = json.load(open('results/m3_3head.json'))
print(f"test period {run['split']['test_from']}+   n = {run['n_test']:,}")
print(f"trained on {run['n_train']:,} rows in {run['train_seconds'] / 60:.1f} min on {run['device']}\n")
for head in run['heads']:
    s = run['test'][head]
    print(f"  {head:12s} accuracy {s['accuracy']:.4f}   macro-F1 {s['macro_f1']:.4f}")

high = run['test']['materiality']['per_class']['high']
print(f"\n  high-materiality: recall {high['recall']:.3f}  precision {high['precision']:.3f}"
      f"  (n={int(high['support'])})")
print(f"  teacher self-consistency 93.0% <- the ceiling")
print(f"  inference {run['ms_per_headline']:.2f} ms/headline on {run['device']}")

## 5. Bring the results home

Downloads the results JSON so it can be committed. The trained weights stay here — they are ~440 MB
and git-ignored; re-run this notebook to reproduce them rather than storing them.

**Colab disconnects after idling and wipes the session**, so download before closing the tab.

In [ ]:
from google.colab import files

files.download('results/m3_3head.json')

## Next, once the main number looks sane

These are the remaining M3 runs. Each is the same command with one flag changed, and each takes
about the same five minutes:

```bash
# does the direction head help or hurt the other two?
python train_student.py --labels full --sample ../$SAMPLE --no-direction --tag no_direction

# learning curve: how many labels did we actually need?
for n in 1000 2500 5000 10000 20000; do
  python train_student.py --labels full --sample ../$SAMPLE --limit $n
done

# smaller, faster student for the size comparison
python train_student.py --labels full --sample ../$SAMPLE \
    --encoder distilbert-base-uncased --tag distilbert
```

The learning curve is the one that decides whether labelling the remaining corpus would buy
anything — if the curve is flat from 10k to 20k, more labels are not the constraint.